# AG11 · Multi-Agent Systems

**Track:** Agents (lesson 11 of 15) · **Time:** ~2.5 h · **Mode:** runs OFFLINE for free, or LIVE with `ANTHROPIC_API_KEY`
**Prereqs:** AG02 (agent loop), AG05 (context engineering), AG07 (workflow patterns), AG09 (guardrails) · **You will build:** a lead + two workers research team, with token accounting against a single-agent baseline

---
## 🎯 Why this matters
"One agent is good, so ten must be better" is the most expensive intuition in agent engineering. Multi-agent systems *can* win — when work is
parallel, when one context window would overflow, or when different roles need different tools and permissions — but every extra agent
adds tokens, latency, and a new place for information to get lost in translation.
Your job is to know the patterns **and** to be able to say, with numbers, "no, one agent is enough here".

**Real scenario.** *Kiwi Grid Partners* is deciding whether to buy into one of two EV-charging networks, **ChargeNZ** or **VoltWay**.
An analyst needs site counts, utilisation and pricing for each, then a comparison. The raw filings are long. Do we hand it to one agent,
or to a lead who delegates one company to each of two workers?

## 🧭 Learning objectives
By the end you can:
1. Distinguish three patterns — **supervisor/subagent**, **handoff**, and **shared state** — and say what each one passes between agents.
2. Build a subagent with an **isolated context** and prove that its raw working never reaches the lead.
3. Account for tokens per agent and compute the **cost multiplier** and **peak context** of a multi-agent run versus a single agent.
4. Explain, with a simple model, *when* splitting wins on context and wall-clock and when it only costs more.
5. Defend against **failure cascades** (a worker's wrong answer trusted by the lead) with structured outputs and verification.

In [ ]:
# Setup: runs LIVE with ANTHROPIC_API_KEY, otherwise OFFLINE (scripted replay, free)
import sys, pathlib, subprocess
_find = lambda: next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "agentkit").is_dir()), None)
_root = _find()
if _root is None:                        # Colab / standalone download: fetch the repo helper package
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/nexageapps/AI.git", "_ai_repo"], check=False)
    _root = pathlib.Path("_ai_repo").resolve()
sys.path.insert(0, str(_root))
from agentkit import setup
setup()

## 1 · Concept: three ways agents can cooperate

```
 SUPERVISOR / SUBAGENT              HANDOFF (control transfer)            SHARED STATE (blackboard)

       ┌────────┐                   ┌────────┐   ┌─────────┐              ┌────────┐ ┌────────┐
       │  lead  │                   │ triage │──▶│ billing │              │ agent A│ │ agent B│
       └─┬────┬─┘                   └────────┘   └─────────┘              └───┬────┘ └───┬────┘
  task ▼ │    │ ▲ short summary      one agent is active at a time,            │ read/write│
    ┌────┴┐ ┌─┴──┐                   the *next* one gets a context            ▼           ▼
    │ w1  │ │ w2 │  (fresh context   packet, not the whole history        ┌──────────────────┐
    └─────┘ └────┘   each)                                                │ notes / DB / files│
                                                                          └──────────────────┘
```

| Pattern | Who holds control? | What crosses the boundary? | Typical use |
|---|---|---|---|
| **Supervisor + subagents** | the lead; workers return and disappear | a *task* down, a *short result* up | parallel research, big-context digestion |
| **Handoff** | moves from agent to agent | a *context packet* (who, what, why) | triage → specialist (billing, legal) |
| **Shared state** | nobody; agents poll/write | records in a shared store | long pipelines, human-visible work queues |

**The one idea that makes subagents worth it is *context isolation*.** A worker may read 40 000 tokens of filings and think out loud for twenty
turns; the lead only ever sees the 300-token finding. The lead's context stays small and focused (AG05), and the worker's context is thrown away.

**The one fact that makes them costly:** every agent carries its own system prompt, tool schemas and re-sent history, and the lead must spend
tokens writing the task and reading the result. Multi-agent is a trade of **tokens for parallelism and context headroom**. We will measure the exchange rate.

## 2 · The world: two competitors' filings
We embed a small corpus (four documents, deterministic, generated with a fixed seed). Each company has an *annual summary* and a *pricing & utilisation*
note whose site table makes the documents long enough to matter for context. Two tools: `search_docs` and `read_doc`.

In [ ]:
import json, math
import numpy as np
from agentkit import tool, ToolBox, get_llm, say, call, calls, run_agent, ReplayLLM, PolicyLLM
from agentkit.llm import DEFAULT_MODEL
from agentkit.trace import PRICES

rng = np.random.default_rng(11)
CITIES = ["Auckland", "Hamilton", "Tauranga", "Wellington", "Christchurch", "Dunedin", "Napier", "Nelson", "Rotorua", "Queenstown"]
PROFILE = {"ChargeNZ": dict(sites=24, util_mean=34, price=0.42, founded=2016),
           "VoltWay":  dict(sites=18, util_mean=47, price=0.49, founded=2019)}

def make_docs():
    docs = {}
    for k, (co, p) in enumerate(PROFILE.items()):
        util = np.clip(rng.normal(p["util_mean"], 9, p["sites"]), 8, 85).round(0).astype(int)
        rows = "\n".join(f"  site {i+1:02d} {CITIES[i % len(CITIES)]:<13} chargers={int(rng.integers(2, 9))} utilisation={u}%"
                         for i, u in enumerate(util))
        docs[f"D{2*k+1}"] = dict(company=co, title=f"{co} annual summary",
            text=(f"{co} annual summary. Founded {p['founded']}. The network operates {p['sites']} fast-charging sites across the North and South Islands. "
                  f"Management reports growth in sessions but notes grid-connection delays. Capital expenditure continues at a measured pace. "
                  f"Directors' statement: the business remains focused on reliability of existing sites before expansion."))
        docs[f"D{2*k+2}"] = dict(company=co, title=f"{co} pricing and utilisation", util=util.tolist(),
            text=(f"{co} pricing and utilisation. Standard tariff: ${p['price']:.2f} per kWh, no subscription. Per-site utilisation (share of hours in use):\n{rows}"))
    return docs

DOCS = make_docs()
print({k: (v["title"], len(v["text"])) for k, v in DOCS.items()})

def truth(company):
    p = PROFILE[company]
    util = next(d["util"] for d in DOCS.values() if d["company"] == company and "util" in d)
    return dict(company=company, sites=p["sites"], avg_utilisation_pct=round(float(np.mean(util)), 1), price_per_kwh=p["price"])

@tool
def search_docs(query: str) -> list:
    """Keyword-search the filings. Returns a list of {doc_id, title} for documents whose title or company matches.

    Args:
        query: A company name or topic, e.g. "VoltWay pricing".
    """
    q = query.lower().split()
    hits = [{"doc_id": i, "title": d["title"]} for i, d in DOCS.items() if any(w in d["title"].lower() or w in d["company"].lower() for w in q)]
    return hits

@tool
def read_doc(doc_id: str) -> str:
    """Return the full text of one document. Documents are long; read only what you need.

    Args:
        doc_id: A document id from search_docs, e.g. "D2".
    """
    if doc_id not in DOCS:
        raise KeyError(f"No document {doc_id}. Valid: {sorted(DOCS)}")
    return DOCS[doc_id]["text"]

WORKER_TOOLS = ToolBox(search_docs, read_doc)
print(truth("ChargeNZ")); print(truth("VoltWay"))

## 3 · Measuring: a token meter around any LLM
Offline replays have no real usage numbers, so we estimate tokens as `characters / 4` (a rough rule of thumb for English; LIVE mode could use
`reply.usage`). Estimates are crude but **consistent**, and comparisons between designs are what we care about.
The `Meter` wraps an LLM and records, for every call, the **input tokens** (system + tool schemas + the whole message history — it is re-sent every turn)
and the output tokens. The largest input is the agent's **peak context**.

In [ ]:
def est_tokens(x) -> int:
    if not x: return 0
    s = x if isinstance(x, str) else json.dumps(x, default=str)
    return max(1, len(s) // 4)

class Meter:
    """Wrap an LLM; record estimated tokens for every call into `ledger`."""
    def __init__(self, llm, name, ledger):
        self.llm, self.name, self.model = llm, name, getattr(llm, "model", "")
        self.in_calls, self.out_tokens = [], 0
        ledger.append(self)
    def chat(self, messages, system="", tools=None, max_tokens=4096, **kw):
        self.in_calls.append(est_tokens(system) + est_tokens(tools) + est_tokens(messages))
        r = self.llm.chat(messages, system=system, tools=tools, max_tokens=max_tokens, **kw)
        self.out_tokens += est_tokens(r.content)
        return r

def summarise(ledger, title):
    rows = [(m.name, len(m.in_calls), sum(m.in_calls), m.out_tokens, max(m.in_calls)) for m in ledger]
    tin, tout = sum(r[2] for r in rows), sum(r[3] for r in rows)
    pin, pout = PRICES[DEFAULT_MODEL]
    cost = (tin * pin + tout * pout) / 1e6
    print(f"== {title}\n{'agent':<18}{'calls':>6}{'input':>9}{'output':>8}{'peak ctx':>10}")
    for r in rows: print(f"{r[0]:<18}{r[1]:>6}{r[2]:>9}{r[3]:>8}{r[4]:>10}")
    print(f"{'TOTAL':<18}{sum(r[1] for r in rows):>6}{tin:>9}{tout:>8}{max(r[4] for r in rows):>10}   est. cost ${cost:.4f} ({DEFAULT_MODEL})")
    return dict(calls=sum(r[1] for r in rows), tin=tin, tout=tout, total=tin + tout, peak=max(r[4] for r in rows), cost=cost, rows=rows)

## 4 · Baseline: one agent does everything
One agent searches both companies (parallel calls), reads all four documents, and writes the comparison. This is the design to beat.

In [ ]:
SYSTEM_SINGLE = ("You are an investment analyst. Use the tools to find facts; never guess numbers. Compare ChargeNZ and VoltWay on "
                 "site count, average site utilisation and price per kWh, citing document ids.")
QUESTION = "Compare ChargeNZ and VoltWay: sites, average utilisation, tariff. Which has the stronger utilisation?"

def final_comparison():
    a, b = truth("ChargeNZ"), truth("VoltWay")
    better = a if a["avg_utilisation_pct"] > b["avg_utilisation_pct"] else b
    return (f"ChargeNZ: {a['sites']} sites, {a['avg_utilisation_pct']}% average utilisation, ${a['price_per_kwh']:.2f}/kWh [D1, D2]. "
            f"VoltWay: {b['sites']} sites, {b['avg_utilisation_pct']}% average utilisation, ${b['price_per_kwh']:.2f}/kWh [D3, D4]. "
            f"{better['company']} has the stronger utilisation.")

LEDGER_SINGLE = []
single_llm = Meter(get_llm(script=[
    calls(("search_docs", {"query": "ChargeNZ"}), ("search_docs", {"query": "VoltWay"})),
    calls(*[("read_doc", {"doc_id": d}) for d in ["D1", "D2", "D3", "D4"]]),
    say(final_comparison()),
]), "single-agent", LEDGER_SINGLE)
res_single = run_agent(single_llm, WORKER_TOOLS, QUESTION, system=SYSTEM_SINGLE, max_turns=8)
print(res_single.final_text)
S1 = summarise(LEDGER_SINGLE, "single agent")
assert res_single.stop == "end_turn" and "VoltWay" in res_single.final_text

## 5 · Build it: lead + two workers (the subagent pattern)
The lead has **one tool**, `delegate(company, question)`. Inside, the tool starts a *fresh* `run_agent` for a worker: new empty message list, its own
system prompt, its own tools. When the worker finishes, only its **final text** is returned to the lead as an ordinary tool result.
That return value is the *entire interface* between the two contexts — which is exactly why you should make it a small, structured contract.

**Design question before you read on:** what should the worker be forced to return so that the lead can trust and check it?
(We answer with a pydantic schema in §7.)

In [ ]:
from pydantic import BaseModel, Field

class Finding(BaseModel):
    """Contract between a worker and the lead: small, typed, and checkable."""
    company: str
    sites: int
    avg_utilisation_pct: float
    price_per_kwh: float
    citations: list[str] = Field(description="ids of documents the numbers come from")

WORKER_SYSTEM = ("You are a research worker for ONE company. Read the filings with the tools and reply with ONLY a JSON object "
                 "{company, sites, avg_utilisation_pct, price_per_kwh, citations[]}. Compute the average yourself from the table. No prose.")
LEAD_SYSTEM = ("You are the lead analyst. Delegate one company to each worker with `delegate`, in parallel. Do not read filings yourself. "
               "Write the final comparison from the workers' findings and cite their document ids.")

def worker_script(company):
    ids = [i for i, d in DOCS.items() if d["company"] == company]
    f = truth(company)
    return [call("search_docs", query=company),
            calls(*[("read_doc", {"doc_id": i}) for i in ids]),
            say(json.dumps({**f, "citations": ids}))]

LEDGER_MULTI = []

@tool
def delegate(company: str, question: str) -> str:
    """Give ONE company's research to a worker agent that has its own fresh context. Returns the worker's JSON finding only, never its raw working.

    Args:
        company: Exactly one of "ChargeNZ" or "VoltWay".
        question: What the worker should find out, in one or two sentences.
    """
    w = Meter(get_llm(script=worker_script(company)), f"worker:{company}", LEDGER_MULTI)
    res = run_agent(w, WORKER_TOOLS, question, system=WORKER_SYSTEM, max_turns=6)
    return res.final_text

lead_llm = Meter(get_llm(script=[
    calls(("delegate", {"company": "ChargeNZ", "question": "Sites, average site utilisation, and tariff per kWh for ChargeNZ."}),
          ("delegate", {"company": "VoltWay", "question": "Sites, average site utilisation, and tariff per kWh for VoltWay."}),
          _say="Two independent lookups - I'll run them in parallel."),
    say(final_comparison()),
]), "lead", LEDGER_MULTI)
LEDGER_MULTI.insert(0, LEDGER_MULTI.pop())     # show the lead first in the table
res_multi = run_agent(lead_llm, ToolBox(delegate), QUESTION, system=LEAD_SYSTEM, max_turns=6)
print(res_multi.final_text)
M1 = summarise(LEDGER_MULTI, "lead + 2 workers")
assert res_multi.stop == "end_turn"

### 5a · Prove the isolation
Isolation is a property you can *assert*, not just hope for. The raw site table (`site 07 ...`) must exist in the workers' histories and **not** in the lead's.
The lead's transcript should contain only the two short findings.

In [ ]:
lead_text = json.dumps(res_multi.messages, default=str)
assert "utilisation=" not in lead_text, "raw filing text leaked into the lead's context"
assert "site 07" not in lead_text
findings = [m for m in res_multi.messages if m["role"] == "user" and isinstance(m["content"], list)][0]["content"]
parsed = [Finding.model_validate_json(b["content"]) for b in findings]     # the contract holds
print("lead received:", [(f.company, f.sites, f.avg_utilisation_pct) for f in parsed])
print("lead context size:", est_tokens(res_multi.messages), "tokens | single agent's final history:", est_tokens(res_single.messages), "tokens")
assert est_tokens(res_multi.messages) < est_tokens(res_single.messages) / 2

## 6 · Read the exchange rate
Put the two runs side by side. Predict first: is the multi-agent run **cheaper** or **more expensive** in total tokens, and what about the **peak context**?

In [ ]:
print(f"{'':<22}{'single':>10}{'multi':>10}{'multi/single':>14}")
for k, label in [("total", "total tokens"), ("calls", "LLM calls"), ("peak", "peak context"), ("cost", "est. cost $")]:
    a, b = S1[k], M1[k]
    print(f"{label:<22}{a:>10.4g}{b:>10.4g}{b/a:>14.2f}x")
lead_row = M1["rows"][0]
print(f"\nlead's own peak context: {lead_row[4]} tokens (single agent's peak: {S1['peak']})")
assert M1["calls"] > S1["calls"] and M1["total"] > S1["total"]   # more calls AND more tokens for the same small task
assert lead_row[4] < S1["peak"]                       # but the lead's window stays small

**What to notice.** On this *small* task the team makes more LLM calls and spends **more total tokens** than one agent (each worker pays for its own system
prompt, tool schemas and three turns), and the critical path is *longer* in serial LLM turns (lead → worker's 3 turns → lead = 5, versus 3).
What you bought is a **small lead context** — useful only if that matters. It does not matter for four documents.
Anthropic's engineering write-up of its multi-agent research system makes the same point in production terms: agents use several times the tokens of chat,
multi-agent systems several times more than that, so the task's value must justify it (see Further reading).

The interesting question is *how that exchange rate changes with task size*. Let's model it.

## 7 · A model of when splitting pays
Let a single agent read `K` documents **one per turn**, each `D` tokens, with a fixed overhead `B` (system + tool schemas). Because history is re-sent every turn,
turn `t` costs `B + (t-1)·D` input tokens, so the total grows **quadratically** in K:  `(K+1)·B + D·K(K+1)/2`.
Split across `W` workers (`k = K/W` docs each) and each worker costs the same formula with `k`, plus a lead that reads `W` short summaries of `S` tokens.
Serial LLM turns: single `K+1`; multi `(k+1) + 2` (workers run in parallel).

*Caveats the model ignores — and which shrink the quadratic term in practice:* prompt caching, context compaction (AG05), and reading several docs per turn.
Treat the shape of the curve as the lesson, not the numbers.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

B, D, S, LEAD_B = 1500, 1200, 300, 1500

def cost_single(K):  return (K + 1) * B + D * K * (K + 1) / 2
def cost_multi(K, W):
    k = math.ceil(K / W)
    return W * ((k + 1) * B + D * k * (k + 1) / 2) + (2 * LEAD_B + W * S + 600)   # + lead: 2 calls, reads W summaries
def turns_single(K): return K + 1
def turns_multi(K, W): return math.ceil(K / W) + 1 + 2

Ks = np.arange(2, 81, 2)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(Ks, [cost_single(k) / 1e3 for k in Ks], label="single agent", lw=2)
for W in (2, 4, 8):
    ax[0].plot(Ks, [cost_multi(k, W) / 1e3 for k in Ks], label=f"{W} workers", lw=1.5)
ax[0].set(xlabel="documents to digest (K)", ylabel="total input tokens (thousands)", title="Total input tokens"); ax[0].legend()
ax[1].plot(Ks, [turns_single(k) for k in Ks], label="single agent", lw=2)
ax[1].plot(Ks, [turns_multi(k, 4) for k in Ks], label="4 workers", lw=1.5)
ax[1].set(xlabel="documents to digest (K)", ylabel="serial LLM turns", title="Latency proxy (critical path)"); ax[1].legend()
plt.tight_layout(); plt.show()

cross = next(k for k in range(2, 200) if cost_multi(k, 4) < cost_single(k))
print(f"with these parameters, 4 workers become cheaper than one agent from K = {cross} documents")
assert cost_multi(4, 2) > cost_single(4) * 0.9          # tiny task: no real saving in tokens
assert cost_multi(60, 4) < cost_single(60) / 2          # big sequential task: splitting wins clearly
assert turns_multi(60, 4) < turns_single(60)

**Reading the plot.** Splitting buys you two things — a shorter critical path and, for long sequential digestion, fewer *total* tokens because no single
history gets re-sent dozens of times — but only above a task-size threshold. Below it you pay a fixed tax per worker. Real systems also use multi-agent
simply because one context window cannot hold the material, which is a hard constraint rather than a cost optimisation.

## 8 · The handoff pattern
In a **handoff** the *conversation* moves: a triage agent decides who should own the customer's problem and passes a **context packet** — not the raw history —
to a specialist. Control transfers (the triage agent is done); there is no lead waiting for a return value.

Two engineering rules: pass an explicit packet (what the customer wants, what is known, why you are transferring) so the specialist starts clean;
and **bound the hops**, because two agents that each think the other is the right owner will ping-pong forever.

In [ ]:
INVOICES = {"INV-2207": dict(customer="Hana", amount=348.00, status="paid_twice", currency="NZD")}
HANDOFF = {}
ALLOWED = {"billing", "triage"}

@tool
def transfer_to(agent: str, reason: str) -> str:
    """Hand the conversation to a specialist agent. Use only when the request is outside your remit.

    Args:
        agent: "billing" for invoice and payment issues.
        reason: One sentence the specialist will read first (what the customer wants and what you already know).
    """
    if agent not in ALLOWED:
        raise ValueError(f"unknown agent {agent!r}; allowed: {sorted(ALLOWED)}")
    HANDOFF.update(to=agent, reason=reason)
    return f"Transferring to {agent}."

@tool
def get_invoice(invoice_id: str) -> dict:
    """Look up an invoice.

    Args:
        invoice_id: e.g. "INV-2207".
    """
    return INVOICES[invoice_id]

AGENTS = {
    "triage":  dict(system="You are the front-desk agent. You cannot access invoices. Transfer billing questions to billing.", tools=ToolBox(transfer_to),
                    script=lambda: [call("transfer_to", agent="billing", reason="Hana says invoice INV-2207 was charged twice and wants it fixed."),
                                    say("Passing you to our billing specialist now.")]),
    "billing": dict(system="You are the billing specialist. Use get_invoice; never guess amounts.", tools=ToolBox(get_invoice),
                    script=lambda: [call("get_invoice", invoice_id="INV-2207"),
                                    say("Hana, I can see INV-2207 ($348.00 NZD) was paid twice. I've flagged the duplicate for refund.")]),
}

def run_with_handoffs(start, task, max_hops=3, agents=AGENTS):
    cur, path, hops = start, [start], 0
    while True:
        HANDOFF.clear()
        spec = agents[cur]
        res = run_agent(get_llm(script=spec["script"]()), spec["tools"], task, system=spec["system"], max_turns=6)
        if not HANDOFF:
            return res, path
        hops += 1
        if hops > max_hops:
            raise RuntimeError(f"handoff loop detected: {' -> '.join(path)}")
        cur = HANDOFF["to"]; path.append(cur)
        task = f"[Handoff from the previous agent] {HANDOFF['reason']}\nContinue and resolve it."   # the context packet

res, path = run_with_handoffs("triage", "Hi, I'm Hana - I was charged twice for invoice INV-2207!")
print("path:", " -> ".join(path)); print(res.final_text)
assert path == ["triage", "billing"] and "paid twice" in res.final_text
packet = res.messages[0]["content"]
assert "INV-2207" in packet and len(packet) < 200          # the specialist started from a short packet, not the transcript

### 8a · The ping-pong guard
Make both agents "helpfully" transfer to each other. Without a hop limit this never ends; with one it fails loudly.

In [ ]:
pingpong = {
    "triage":  dict(system="", tools=ToolBox(transfer_to), script=lambda: [call("transfer_to", agent="billing", reason="not mine"), say("ok")]),
    "billing": dict(system="", tools=ToolBox(transfer_to), script=lambda: [call("transfer_to", agent="triage", reason="not mine either"), say("ok")]),
}
try:
    run_with_handoffs("triage", "anything", max_hops=3, agents=pingpong)
    raise AssertionError("expected a handoff loop to be detected")
except RuntimeError as e:
    print("caught:", e)
    assert "handoff loop" in str(e)

## 9 · Break it: a failure cascade
The lead trusts what workers say. If a worker returns a confident wrong answer — a misread table, a fabricated citation — the lead builds the final answer on sand,
and nobody downstream sees the original documents. This is *the* characteristic multi-agent failure: **errors compound across hand-offs** and the evidence is isolated away.

Defences, cheapest first: (1) a typed contract (`Finding`) so malformed output is rejected, (2) **citations the lead can check**, (3) recomputing the key number
from the cited source with deterministic code, (4) an independent second worker for high-stakes claims.

In [ ]:
def verify_finding(f: Finding) -> list[str]:
    """Deterministic check of a worker's claim against the cited documents. Returns a list of problems (empty = OK)."""
    problems = []
    for c in f.citations:
        if c not in DOCS: problems.append(f"citation {c} does not exist")
    cited = [DOCS[c] for c in f.citations if c in DOCS]
    if any(d["company"] != f.company for d in cited): problems.append("citation belongs to a different company")
    util_docs = [d for d in cited if "util" in d]
    if not util_docs: problems.append("no cited document contains the utilisation table")
    else:
        real = float(np.mean(util_docs[0]["util"]))
        if abs(real - f.avg_utilisation_pct) > 0.5: problems.append(f"avg utilisation {f.avg_utilisation_pct} != {real:.1f} recomputed from {f.citations}")
    return problems

good = Finding(**{**truth("VoltWay"), "citations": ["D3", "D4"]})
sloppy = Finding(**{**truth("VoltWay"), "avg_utilisation_pct": 61.0, "citations": ["D3", "D9"]})     # confident, wrong, mis-cited
print("good  :", verify_finding(good)); print("sloppy:", verify_finding(sloppy))
assert verify_finding(good) == []
assert len(verify_finding(sloppy)) >= 2

try:
    Finding.model_validate_json('{"company": "VoltWay", "sites": "about eighteen"}')
except Exception as e:
    print("schema rejected a prose answer:", type(e).__name__)

**Wire it in.** The lead (or a thin wrapper around `delegate`) runs `verify_finding` on every result and, on failure, sends the worker back with the specific problem — the
same recover-from-the-error pattern as AG02's `is_error` results. Note the check uses *deterministic code over the source*, not another LLM's opinion.

## 10 · When NOT to use multi-agent (the honest section)
Default to **one agent with good tools** (AG03) and a good workflow (AG07). Reach for several agents only when a specific need below is real and you have *measured* that one agent falls short.

| Situation | Verdict | Why |
|---|---|---|
| Task fits one context window and steps are sequential | **One agent** | Splitting adds tokens and a lossy summary at every boundary |
| Subtasks **depend** on each other's details (e.g. editing the same code) | **One agent** | Workers lack each other's context; conflicting edits and duplicated work |
| Latency matters and the task is short | **One agent** | The critical path gets *longer* (§6) |
| Many independent subtasks, each reading lots of material (research, triage of 50 documents) | **Multi-agent** | Parallelism + isolation keep contexts small |
| Different roles need different **permissions/tools** (read-only researcher vs. the one agent allowed to send email) | **Multi-agent** | Least privilege (AG09) is easier with separate agents |
| Material cannot fit in one window even after compaction | **Multi-agent** | Hard constraint |

A first-pass checklist as code — useful as a design-review prompt, not as an oracle:

In [ ]:
def multi_agent_verdict(independent_subtasks: int, tokens_per_subtask: int, subtasks_share_state: bool, window: int = 200_000, latency_critical: bool = False):
    reasons = []
    if subtasks_share_state: return "single agent", ["subtasks share mutable state / depend on each other's details"]
    total = independent_subtasks * tokens_per_subtask
    if independent_subtasks < 3 and total < window // 4: return "single agent", ["few subtasks and everything fits comfortably in one context"]
    if latency_critical and total < window // 4: return "single agent", ["latency-critical and small: the critical path gets longer"]
    if total >= window // 2: reasons.append("material approaches the context window")
    if independent_subtasks >= 3: reasons.append("many independent subtasks can run in parallel")
    return ("multi-agent", reasons) if reasons else ("single agent", ["no measured need"])

CASES = {
    "compare 2 filings (this lesson)": dict(independent_subtasks=2, tokens_per_subtask=3_000, subtasks_share_state=False),
    "refactor one module across 6 files": dict(independent_subtasks=6, tokens_per_subtask=4_000, subtasks_share_state=True),
    "triage 40 support tickets with long threads": dict(independent_subtasks=40, tokens_per_subtask=15_000, subtasks_share_state=False),
    "answer a refund question": dict(independent_subtasks=1, tokens_per_subtask=2_000, subtasks_share_state=False, latency_critical=True),
}
for name, kw in CASES.items():
    v, why = multi_agent_verdict(**kw); print(f"{name:<46} -> {v:<13} {why}")
assert multi_agent_verdict(**CASES["compare 2 filings (this lesson)"])[0] == "single agent"
assert multi_agent_verdict(**CASES["triage 40 support tickets with long threads"])[0] == "multi-agent"
assert multi_agent_verdict(**CASES["refactor one module across 6 files"])[0] == "single agent"

Note the uncomfortable result: **the lab we just built is a case where multi-agent is the wrong call.** We built it anyway because you learn the mechanics on a small example,
and then you measure. In your own projects, run the single-agent baseline *first* (exactly as in §4) and let the numbers justify the split.

## 11 · Shared state, and talking to *other people's* agents (overview)
**Shared state.** Agents coordinate through a store rather than by calling each other: a task queue, a notes file, a database table. It is simple to inspect and
to resume (AG08), and humans can read it; the price is that you must design for concurrent writes and stale reads. Rule of thumb: give each agent **ownership of
distinct keys** and make writes append-only where you can.

**Agent-to-Agent (A2A).** Everything above happens inside one program you control. A2A is an *open protocol* for agents built by different teams or vendors to
discover each other (via a published "agent card" describing capabilities) and exchange tasks over the network. How it relates to what you know: **MCP (AG10) connects an
agent to tools and data; A2A connects an agent to other agents.** We do not implement it here; if you need cross-organisation agents, read the current specification
before designing anything — the protocol is young and still evolving.

## 12 · Exercises
**Practice (30 min)**
1. Add a third company, `GridGo`, to `PROFILE` (rebuild `DOCS`) and extend the lead's script to delegate three workers. Assert the lead's context still contains no raw table text, and print the new multiplier versus a single agent.
2. In the §7 model, change `D` (doc size) from 1200 to 300. How does the crossover `K` move? Explain why in one sentence.

**Stretch (60 min)**
3. Wrap `delegate` so that it runs `verify_finding` and, if problems are found, **re-runs the worker once** with the problems appended to the question. Test it with a worker script that is wrong the first time and right the second.
4. Add a `max_total_tokens` budget to the lead: using the `Meter` ledger, make `delegate` refuse (return an error string) when the team has already spent more than the budget.

**Challenge (2 h, LIVE recommended)**
5. Run the single-agent and multi-agent designs LIVE on a 10-document corpus you generate. Report total tokens, peak context and wall-clock for each, and decide with data which to ship. Does the crossover you observe match the §7 model's prediction?

In [ ]:
# Starter for exercise 4 - a budget guard shared by every worker. Fill in and enable the asserts.
TEAM_BUDGET = 12_000   # estimated tokens

def team_tokens(ledger) -> int:
    # TODO: sum input + output tokens over every Meter in `ledger`
    raise NotImplementedError

# def guarded_delegate(company, question):
#     if team_tokens(LEDGER_MULTI) > TEAM_BUDGET:
#         return "Error: team token budget exhausted; answer with what you have."
#     return delegate.fn(company, question)
#
# assert team_tokens(LEDGER_MULTI) > 0

## ✅ Checkpoint
1. What exactly crosses the boundary between a lead and a subagent, and why is that the source of both the benefit and the main failure mode?
2. A task has 3 sequential steps that each need the previous step's full output. Single agent or multi-agent? Why?
3. In §6 the multi-agent run used more total tokens *and* more serial turns than the single agent. Name one situation in which it would still be the right design.
4. How would you stop a triage agent and a specialist from bouncing a customer back and forth?

<details><summary>Answers</summary>

1. A task goes down and a (hopefully short, structured) result comes up; the worker's raw reading and reasoning stay behind. That isolation keeps the lead's context small, but it also hides the evidence, so a wrong summary propagates unchecked unless you add citations and verification.
2. Single agent. The steps depend on each other's full details, so a worker would need the entire previous output anyway; you would pay overhead and lose information in the summary.
3. Many independent subtasks that each digest a lot of material (so the single history would be huge and re-sent every turn), or a need for different tool permissions per role, or material that cannot fit in one window.
4. Bound the number of hops, allow-list the transfer targets, pass an explicit context packet containing *why* the transfer happens, and fail loudly (or escalate to a human) when the bound is hit.
</details>

## 📌 Key takeaways
- Three patterns: **subagents** (task down, summary up), **handoffs** (control moves with a context packet), **shared state** (coordination through a store).
- The benefit is **context isolation and parallelism**; the cost is a per-agent token tax, more serial turns on small tasks, and lossy boundaries.
- Measure with a ledger: total tokens, calls, peak context, critical-path turns. Compare against a single-agent baseline *before* adding agents.
- Make the worker-to-lead interface a **typed, citable contract** and verify key numbers with deterministic code.
- Bound hops, budgets and fan-out. "One agent with good tools" is the default; multi-agent is an optimisation you must justify.

## 📚 Further reading
- Anthropic Engineering, *How we built our multi-agent research system* (2025) — anthropic.com/engineering/multi-agent-research-system
- Anthropic Engineering, *Building effective agents* (2024) — anthropic.com/engineering/building-effective-agents
- Google, *Agent2Agent (A2A) protocol* — the project's public specification (check the current version before relying on details)
- Wu et al., *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation* (2023), arXiv:2308.08155

**Next → AG12 · Coding Agents** (a mini coding agent that fixes seeded bugs in a sandboxed repo and is graded by pytest)